# Multi-GPU & Flash Attention Inference

This notebook is optimized for high-throughput inference on 3x RTX 3090 (24GB) GPUs.

**Optimizations:**
1. **Flash Attention 2**: Uses optimized CUDA kernels for attention (requires Ampere+ GPUs).
2. **Multi-GPU**: Uses `accelerate` (via `device_map="auto"`) to distribute the model/memory.
3. **Large Batch Size**: Enabled by the large unified VRAM pool (72GB total).
4. **Directory Processing**: Handles entire folders of data at once.

In [ ]:
!pip install -qU transformers accelerate bitsandbytes
!pip install -qU flash-attn --no-build-isolation

In [ ]:
import os
import json
import glob
import copy
import torch
from tqdm import tqdm
from transformers import AutoTokenizer, AutoModelForCausalLM

print(f"GPUs available: {torch.cuda.device_count()}")
for i in range(torch.cuda.device_count()):
    print(f"GPU {i}: {torch.cuda.get_device_name(i)}")

In [ ]:
MODEL_ID = "Qwen/Qwen3-4B-Instruct-2507"

def load_optimized_model():
    print(f"Loading {MODEL_ID}...")
    
    tokenizer = AutoTokenizer.from_pretrained(MODEL_ID, trust_remote_code=True)
    
    if tokenizer.eos_token_id is None:
        if hasattr(tokenizer, "eod_id"): tokenizer.eos_token = tokenizer.decode(tokenizer.eod_id)
        else: tokenizer.add_special_tokens({'eos_token': '<|endoftext|>'})
        
    if tokenizer.pad_token is None:
        tokenizer.pad_token = tokenizer.eos_token
        
    tokenizer.padding_side = "left"

    model = AutoModelForCausalLM.from_pretrained(
        MODEL_ID,
        device_map="auto",                      
        attn_implementation="flash_attention_2",  
        torch_dtype=torch.float16,              
        trust_remote_code=True
    )
    
    model.generation_config.pad_token_id = tokenizer.pad_token_id
    model.generation_config.eos_token_id = tokenizer.eos_token_id
    
    print("Model loaded successfully with Flash Attention 2.")
    return model, tokenizer

model, tokenizer = load_optimized_model()

In [ ]:
SYSTEM_PROMPT = """You are an Aspect-Based Sentiment Analysis assistant. 
Given a Text and an Aspect term, determine the Sentiment (Positive, Negative, or Neutral) and extract the specific Opinion word or phrase describing that aspect.

Format your response exactly as:
Sentiment: <Positive/Negative/Neutral>
Opinion: <Opinion Phrase>

Examples:
Text: The food was great but the service was terrible.
Aspect term: food
Sentiment: Positive
Opinion: great

Text: Овощной салат также пришёлся по вкусу.
Aspect term: Овощной салат
Sentiment: Positive
Opinion: пришёлся по вкусу"""

def process_directory(input_dir, output_dir, batch_size=64):
    if not os.path.exists(output_dir):
        os.makedirs(output_dir)
        
    files = glob.glob(os.path.join(input_dir, "*.jsonl")) + glob.glob(os.path.join(input_dir, "**/*.jsonl"), recursive=True)
    files += glob.glob(os.path.join(input_dir, "*.json")) + glob.glob(os.path.join(input_dir, "**/*.json"), recursive=True)
    files = sorted(list(set(files)))
    
    print(f"Found {len(files)} files to process.")
    
    for fpath in files:
        print(f"\nReading: {fpath}")
        
        data = []
        try:
            with open(fpath, 'r', encoding='utf-8') as f:
                for line in f:
                    if line.strip(): data.append(json.loads(line))
        except Exception as e:
            print(f"Error reading {fpath}: {e}")
            continue
            
        if not data:
            print("Empty file, skipping.")
            continue
        
        work_items = []
        for entry in data:
            text = entry.get('Text', '')
            
            targets = []
            if 'Aspect_VA' in entry: 
                targets = entry['Aspect_VA']
            elif 'Quadruplet' in entry:
                targets = entry['Quadruplet']
            
            for item in targets:
                raw_aspect = item.get('Aspect', 'NULL')
                if raw_aspect == "NULL" and 'Category' in item:
                    raw_aspect = item['Category'].split('#')[0]
                
                work_items.append({
                    "text": text,
                    "aspect": raw_aspect,
                    "container": item 
                })

        if not work_items:
            print("No work items found in file.")
            continue
            
        print(f"Processing {len(work_items)} items...")
        
        terminators = [tokenizer.eos_token_id]
        try:
            eot = tokenizer.convert_tokens_to_ids("<|eot_id|>")
            if isinstance(eot, int): terminators.append(eot)
        except: pass

        for i in tqdm(range(0, len(work_items), batch_size), desc=f"Inferencing {os.path.basename(fpath)}"):
            batch = work_items[i:i + batch_size]
            batch_texts = []
            
            for item in batch:
                msgs = [
                    {"role": "system", "content": SYSTEM_PROMPT},
                    {"role": "user", "content": f"Text: {item['text']}\\nAspect term: {item['aspect']}"}
                ]
                batch_texts.append(tokenizer.apply_chat_template(msgs, tokenize=False, add_generation_prompt=True))
            
            inputs = tokenizer(
                batch_texts,
                return_tensors="pt",
                padding=True,
                truncation=True,
                max_length=1024
            ).to(model.device)
            
            with torch.no_grad():
                outputs = model.generate(
                    **inputs,
                    max_new_tokens=64,
                    eos_token_id=terminators,
                    pad_token_id=tokenizer.pad_token_id,
                    do_sample=True,
                    temperature=0.3,
                    top_p=0.9
                )
            
            new_tokens = outputs[:, inputs.input_ids.shape[1]:]
            decoded = tokenizer.batch_decode(new_tokens, skip_special_tokens=True)
            
            for work_item, resp_text in zip(batch, decoded):
                sentiment = "Unknown"
                opinion = "Unknown"
                for line in resp_text.strip().split('\n'):
                    if line.startswith("Sentiment:"): sentiment = line.split(":", 1)[1].strip()
                    elif line.startswith("Opinion:"): opinion = line.split(":", 1)[1].strip()
                
                work_item['container']['Gen_Sentiment'] = sentiment
                work_item['container']['Gen_Opinion'] = opinion
        
        out_name = os.path.basename(fpath)
        out_full_path = os.path.join(output_dir, out_name)
        
        with open(out_full_path, 'w', encoding='utf-8') as f_out:
            for entry in data:
                f_out.write(json.dumps(entry, ensure_ascii=False) + "\n")
        
        print(f"Saved processed file to: {out_full_path}")

In [ ]:
INPUT_DIRECTORY = "/kaggle/working/dimabsa/dimabsa_marked_dev" 
OUTPUT_DIRECTORY = "/kaggle/working/dimabsa/processed_data_results"

process_directory(INPUT_DIRECTORY, OUTPUT_DIRECTORY, batch_size=64)